# 🔍 G-LRAG Retrieval — Google Colab Edition

Run the **G-LRAG hybrid retrieval** pipeline (lexical BM25/FTS5 + optional
dense FAISS + optional graph expansion) on Colab with an **input query**.

This notebook is self-contained: it clones the retrieval source code from
GitHub and loads the data artifacts from your Google Drive (Colab's disk is
ephemeral, so the big artifacts must live on Drive, not the repo).

---

## 📦 Input data you must provide (on Google Drive)

Put these files in a Google Drive folder, e.g.
`/content/drive/MyDrive/Road2AI_ApplePie/data/stage6_data/`.
Set `DATA_DIR` in the Setup cell to that folder.

| File | Size | Required? | Purpose |
|---|---|---|---|
| `chunk_store.sqlite` | ~2.7 GB | ✅ **Always** | Lexical BM25/FTS5 index + full `chunk_text` + metadata (`chunks`, `chunks_fts` tables) |
| `faiss_index__BAAI_bge-m3.index` | ~2.4 GB | ⛔ Only for dense leg | FAISS `IndexFlatIP` (636,585 × 1024, L2-normalised) |
| `chunk_meta_slim.parquet` | ~86 MB | ⛔ Only for dense leg | Metadata sidecar aligning FAISS position → `row_idx` |
| `embed_model_meta__BAAI_bge-m3.json` | 156 B | ⛔ Only for dense leg | Bundle-compat check (model name, dim, count) |
| `kg.gpickle` | ~tens of MB | ⛔ Only for graph expand | `networkx.MultiDiGraph` knowledge graph |
| `dev_set/questions.json` | ~3.5 KB | optional | 20 example dev-set questions |
| `dev_set/ground_truth.json` | ~8.3 KB | optional | Gold answers for the optional F2 evaluation cell |

**Minimum to run a lexical-only query:** just `chunk_store.sqlite` (~2.7 GB).

**Full hybrid:** all four Stage-6 files (~5.2 GB) + optionally `kg.gpickle`.

> 💡 Where to get them: they were produced by your local Stage-6 pipeline
> (`Road2AI_ApplePie/data/stage6_data/`) and are git-ignored (too big).
> Upload them to Drive once and reuse across Colab sessions.

---

## ⚙️ What the notebook needs from Colab
- **Runtime type:** `T4 GPU` (or any GPU runtime) — **required**. This
  notebook is GPU-first: the dense leg runs `BAAI/bge-m3` via FlagEmbedding
  and the reranker runs `BAAI/bge-reranker-v2-m3`, both on CUDA. Set it via
  *Runtime → Change runtime type → T4 GPU* before running anything.
- **Disk:** the artifacts total ~5.2 GB; the default ~100 GB Colab disk is fine.
- **RAM:** standard Colab (~12 GB) works; `bm25_ranked` is the heaviest step.

## 1. Setup — clone repo, mount Drive, install deps, configure

In [ ]:
# ===== Configuration (edit these) =====================================
GITHUB_REPO = "https://github.com/vkb0205/Road2AI_ApplePie.git"
REPO_BRANCH = "main"
REPO_DIR    = "/content/Road2AI_ApplePie"

# Folder on your Google Drive that holds the Stage-6 artifacts + dev_set.
# Mirror the repo layout:  .../stage6_data/*.  and  .../dev_set/*
DATA_DIR = "/content/drive/MyDrive/Road2AI_ApplePie/data/stage6_data"
DEV_DIR  = "/content/drive/MyDrive/Road2AI_ApplePie/dev_set"

# Pipeline switches ------------------------------------------------------
# This notebook is GPU-first: dense + rerank legs are ON by default and
# run on CUDA. (Graph expansion stays off unless you also upload kg.gpickle.)
USE_DENSE = True    # load FAISS + BGE-m3 query encoder (runs on GPU + ~2.4GB)
USE_GRAPH = False   # True -> load kg.gpickle for graph expansion
USE_RERANK = True   # cross-encoder rerank (BAAI/bge-reranker-v2-m3, on GPU)
FTS_MODE = "bm25_ranked"   # 'bm25_ranked' (baseline) or 'fts_fast' (faster, weaker)
# ======================================================================

In [ ]:
# --- 0. GPU runtime check (fail fast) ---------------------------------
# This notebook is GPU-first. torch is installed below; here we just probe
# the NVIDIA driver via nvidia-smi to confirm a GPU runtime is attached.
import subprocess, sys
gpu_ok = subprocess.run("nvidia-smi", shell=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
if not gpu_ok:
    raise SystemExit(
        "❌ No GPU detected. This notebook is configured for a GPU runtime.\n"
        "   In Colab: Runtime → Change runtime type → T4 GPU, then restart and run all."
    )
!nvidia-smi -L
print("[gpu] GPU runtime confirmed.")

In [ ]:
import os, sys, time, shutil
from pathlib import Path

# --- 1a. Clone the retrieval source code (lightweight, ~MB) -----------
if not Path(REPO_DIR).exists():
    print(f"[clone] {GITHUB_REPO} -> {REPO_DIR}")
    !git clone --depth 1 -b {REPO_BRANCH} {GITHUB_REPO} {REPO_DIR}
else:
    print(f"[clone] {REPO_DIR} already present")

SRC = Path(REPO_DIR) / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print("[src on path]", SRC)

# --- 1b. Mount Google Drive for the big artifacts ----------------------
from google.colab import drive
drive.mount('/content/drive')

DATA = Path(DATA_DIR)
DEV  = Path(DEV_DIR)
print("[DATA_DIR exists]", DATA.exists(), DATA)
print("[DEV_DIR  exists]", DEV.exists(),  DEV)

In [ ]:
# --- 1c. Install dependencies -----------------------------------------
# Light deps are always needed (pandas/pyarrow/networkx are usually
# preinstalled on Colab; we ensure them). Heavy GPU deps only when the
# corresponding leg is enabled.
!pip -q install pandas pyarrow networkx pyyaml python-dotenv

if USE_DENSE or USE_RERANK:
    print("[install] dense/rerank GPU deps (faiss-cpu, FlagEmbedding, torch+CUDA) ...")
    # CUDA 12.1 wheels match Colab's default CUDA runtime; adjust if Colab
    # changes its base image. `faiss-cpu` is used because the Stage-6 index is
    # IndexFlatIP (GPU not needed for the search itself; the GPU is for the
    # BGE-m3 query encoder + the cross-encoder reranker).
    !pip -q install "faiss-cpu>=1.7.4" "FlagEmbedding>=1.2.10" "numpy>=1.24"
    !pip -q install torch --index-url https://download.pytorch.org/whl/cu121
    # Verify CUDA is actually available from the installed torch.
    import torch as _t
    print("[torch]", _t.__version__, "| cuda available:", _t.cuda.is_available(),
          "| device:", _t.cuda.get_device_name(0) if _t.cuda.is_available() else "CPU")
    assert _t.cuda.is_available(), (
        "torch installed but CUDA not available. Make sure the Colab runtime "
        "is a GPU runtime (Runtime → Change runtime type → T4 GPU)."
    )

print("[deps] ready")

## 2. Verify your input data is present

Run this to confirm the files you put on Drive are found before loading.

In [ ]:
required = {"chunk_store.sqlite": DATA}
if USE_DENSE:
    required.update({
        "faiss_index__BAAI_bge-m3.index": DATA,
        "chunk_meta_slim.parquet": DATA,
        "embed_model_meta__BAAI_bge-m3.json": DATA,
    })
if USE_GRAPH:
    required["kg.gpickle"] = DATA   # or move it wherever; adjust below if needed

ok = True
for name, d in required.items():
    p = d / name
    mb = (p.stat().st_size / 1e6) if p.exists() else 0
    flag = "OK " if p.exists() else "MISSING"
    if not p.exists(): ok = False
    print(f"{flag}  {p}  ({mb:.1f} MB)")

if DEV.exists():
    print(f"OK   {DEV/'questions.json'}")
    print(f"OK   {DEV/'ground_truth.json'}")
else:
    print(f"(optional dev_set not found at {DEV})")

assert ok, "❌ Missing required input files — upload them to DATA_DIR on Drive (see the top table)."

## 3. Build the retriever

Constructs the [`HybridRetriever`](src/retrieval/retriever.py) with the legs
you enabled. Lexical leg is always on; dense + graph + rerank are optional.

In [ ]:
from retrieval.bm25_index import FTSIndex
from retrieval.faiss_index import FAISSIndex, BGEQueryEncoder
from retrieval.graph_expand import GraphExpander
from retrieval.retriever import HybridRetriever, RetrievalConfig, make_relevant_lists

DB         = DATA / "chunk_store.sqlite"
FAISS_IDX  = DATA / "faiss_index__BAAI_bge-m3.index"
META       = DATA / "chunk_meta_slim.parquet"
MODEL_META = DATA / "embed_model_meta__BAAI_bge-m3.json"
KG         = DATA / "kg.gpickle"

# --- lexical leg (always on) ------------------------------------------
t0 = time.time()
fts = FTSIndex(str(DB), mode=FTS_MODE).open()
print(f"[fts] rows={fts.n_rows:,}  backend={fts.lexical_backend!r}  mode={FTS_MODE!r}  ({time.time()-t0:.1f}s)")

# --- GPU device used by the dense/rerank legs -------------------------
if USE_DENSE or USE_RERANK:
    import torch
    _dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[gpu] active device = {_dev}" + (f" ({torch.cuda.get_device_name(0)})" if _dev.type=='cuda' else ""))

# --- dense leg (optional, GPU) ----------------------------------------
faiss_index = None
query_encoder = None
if USE_DENSE:
    t0 = time.time()
    faiss_index = FAISSIndex(str(FAISS_IDX), str(META), str(MODEL_META)).load_index()
    print(f"[dense] faiss ntotal={faiss_index.ntotal:,} dim={faiss_index.dim} ({time.time()-t0:.1f}s)")
    # BGE-m3 runs on CUDA via FlagEmbedding; fp16 on GPU for speed.
    query_encoder = BGEQueryEncoder("BAAI/bge-m3", use_fp16=torch.cuda.is_available())
    print("[dense] BGE-m3 query encoder ready (downloads weights on first encode)")

# --- graph expander (optional) ----------------------------------------
graph_expander = None
if USE_GRAPH and KG.exists():
    t0 = time.time()
    graph_expander = GraphExpander.from_graph_and_meta(str(KG), str(META))
    print(f"[graph] expander loaded ({time.time()-t0:.1f}s)")
elif USE_GRAPH:
    print(f"[graph] kg.gpickle not found at {KG}; graph expansion disabled")

cfg = RetrievalConfig(
    use_dense=faiss_index is not None,
    use_reranker=USE_RERANK,
    top_bm25=50,
    top_dense=50,
    fused_top=30,
    expanded_top=50,
    final_top_k=5,
)
retriever = HybridRetriever(fts, faiss_index=faiss_index, graph_expander=graph_expander,
                            query_encoder=query_encoder, config=cfg)
print("[retriever] ready")

## 4. Run a query

Edit `QUERY` and run. Returns the final top-K hits with metadata + chunk text,
plus the collapsed `relevant_docs` / `relevant_articles`.

In [ ]:
# ──────────────────────────  EDIT YOUR QUERY HERE  ──────────────────────────
QUERY = "Thủ tục đăng ký doanh nghiệp lần đầu bao gồm những bước nào?"
# ────────────────────────────────────────────────────────────────────────────

TEXT_PREVIEW = 800   # chars of chunk_text to show per hit (0 = full text)

t0 = time.time()
hits = retriever.retrieve(QUERY, fetch_text=True)
print(f"Retrieved {len(hits)} hits in {time.time()-t0:.2f}s\n")

for i, h in enumerate(hits, 1):
    print(f"#{i}  row_idx={h.row_idx}  score={h.score:+.4f}  source={h.source}")
    print(f"    law_id    : {h.law_id}")
    print(f"    ten_van_ban: {h.ten_van_ban}")
    print(f"    dieu_so   : {h.dieu_so}")
    txt = h.chunk_text or ""
    if TEXT_PREVIEW and len(txt) > TEXT_PREVIEW:
        txt = txt[:TEXT_PREVIEW] + " …[truncated]"
    print(f"    chunk_text:\n{txt}")
    print("-" * 100)

docs, articles = make_relevant_lists(hits)
print("\nrelevant_docs:", docs)
print("relevant_articles:", articles)

## 5. (Optional) Load a dev-set example question

If you uploaded `dev_set/questions.json`, pick one of the 20 questions by index.

In [ ]:
import json
if DEV.exists() and (DEV / "questions.json").exists():
    QUESTIONS = json.loads((DEV / "questions.json").read_text(encoding="utf-8"))
    for q in QUESTIONS:
        print(f"{q['id']:>2}. {q['question']}")
else:
    print(f"dev_set not found at {DEV}. Upload dev_set/questions.json to use the picker.")
    QUESTIONS = []

In [ ]:
PICK = 1   # 1..20
if QUESTIONS:
    QUERY = QUESTIONS[PICK - 1]["question"]
    print("QUERY set to:\n", QUERY)
else:
    print("No dev questions loaded; set QUERY manually in the cell above.")

## 6. (Optional) Batch-run the whole dev set + F2 evaluation

Runs retrieval for all 20 dev questions and scores with the repo's
[`dev_set/eval.py`](src/../dev_set/eval.py) F2 macro metric.
Requires `dev_set/questions.json` + `dev_set/ground_truth.json`.

In [ ]:
import sys as _sys, json as _json
from types import SimpleNamespace

DEV_LOCAL = Path(REPO_DIR) / "dev_set"
# Prefer Drive copy if present (in case you edited it), else the cloned repo copy.
QPATH = (DEV / "questions.json") if (DEV / "questions.json").exists() else (DEV_LOCAL / "questions.json")
GTPATH = (DEV / "ground_truth.json") if (DEV / "ground_truth.json").exists() else (DEV_LOCAL / "ground_truth.json")
print("questions:", QPATH, "| ground_truth:", GTPATH)

questions = _json.loads(QPATH.read_text(encoding="utf-8"))
records, t0 = [], time.time()
for q in questions:
    hits = retriever.retrieve(q["question"], fetch_text=False)
    docs, articles = make_relevant_lists(hits)
    records.append({"id": q["id"], "question": q["question"], "answer": "",
                    "relevant_docs": docs, "relevant_articles": articles})
print(f"Ran {len(records)} questions in {time.time()-t0:.1f}s")

OUT = Path("/content/results_colab.json")
OUT.write_text(_json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

# Score with the repo's evaluator.
_sys.path.insert(0, str(DEV_LOCAL.parent))  # so `import dev_set.eval` works
from dev_set.eval import f2_macro
gt = _json.loads(GTPATH.read_text(encoding="utf-8"))
print(f"\nF2 macro = {f2_macro(records, gt):.4f}")
print("results written to", OUT)

## 7. Cleanup

In [ ]:
fts.close()
print("FTS index closed. Done.")